WORKSHOP ON RAG 


RAG steps : 

1. **INGESTION**:
2. **EMBEDDING**:
3. **RETRIEVAL**:
4. **AUGMENTATION**:
5. **GENERATE**: 


Typical use cases that will be explored in this workshop : Banque / assurance - Health and medical applications - Agent with product manuals …

In [1]:
# Packages import 
from __future__ import annotations


import hashlib
import io
import os
import re
import time 
import uuid
from collections import Counter
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple
import fitz

from transformers import AutoTokenizer, AutoModelForCausalLM
from pytest import Parser
import torch
import numpy as np
import faiss
from typing import List, Dict, Any, Optional, Tuple, Iterable
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi


In [2]:
# Data Ingestion / preprocessing

@dataclass(frozen=True)
class Document:
    doc_id: str
    text: str
    metadata: Optional[Dict[str, Any]] = None


@dataclass(frozen=True)
class Chunk: 
    chunk_id: str
    doc_id: str
    text: str
    metadata: Optional[Dict[str, Any]] = None
    start: Optional[int] = None
    end: Optional[int] = None



@dataclass(frozen=True)
class RetrievedChunk:
    chunk: Chunk
    score: float
    stage: str = 'dense'


In [3]:
class PdfParser:
    def __init__(
        self,
        *,
        per_page: bool = True,
        header_footer_lines: int = 2,
        header_footer_min_page_ratio: float = 0.6,
        min_text_chars_scanned_like: int = 50,
    ) -> None:
        self.per_page = per_page
        self.header_footer_lines = header_footer_lines
        self.header_footer_min_page_ratio = header_footer_min_page_ratio
        self.min_text_chars_scanned_like = min_text_chars_scanned_like

    def parse(self, source: Any) -> List[Document]:
        pdf_bytes, source_name = self._load_pdf_bytes(source)
        file_hash = hashlib.sha256(pdf_bytes).hexdigest()

        with fitz.open(stream=pdf_bytes, filetype="pdf") as doc:
            num_pages = doc.page_count
            raw_pages: List[str] = []
            page_meta: List[Dict[str, Any]] = []

            for p in range(num_pages):
                page = doc.load_page(p)
                text = page.get_text("text") or ""
                text = self._basic_clean(text)
                raw_pages.append(text)
                is_scanned_like = len(text.strip()) < self.min_text_chars_scanned_like
                page_meta.append(
                    {
                        "source": source_name,
                        "file_hash": file_hash,
                        "page": p + 1,
                        "num_pages": num_pages,
                        "is_scanned_like": is_scanned_like,
                    }
                )

            cleaned_pages = self._remove_repetitive_headers_footers(raw_pages)

            if self.per_page:
                docs: List[Document] = []
                for i, page_text in enumerate(cleaned_pages):
                    doc_id = f"{file_hash}_p{i+1}"
                    docs.append(
                        Document(
                            doc_id=doc_id,
                            text=page_text,
                            metadata=page_meta[i],
                        )
                    )
                return docs
            else:
                joined = "\n\n".join(
                    f"[PAGE {i+1}]\n{t}".strip()
                    for i, t in enumerate(cleaned_pages)
                    if t.strip()
                )
                meta = {
                    "source": source_name,
                    "file_hash": file_hash,
                    "num_pages": num_pages,
                    "per_page": False,
                    "scanned_like_pages": sum(
                        1 for m in page_meta if m["is_scanned_like"]
                    ),
                }
                return [Document(doc_id=file_hash, text=joined, metadata=meta)]

    def _load_pdf_bytes(self, source: Any) -> Tuple[bytes, str]:
        if isinstance(source, (str, os.PathLike)):
            path = str(source)
            with open(path, "rb") as f:
                data = f.read()
            return data, os.path.basename(path)

        if isinstance(source, bytes):
            return source, "bytes.pdf"

        if hasattr(source, "read"):
            data = source.read()
            if isinstance(data, str):
                data = data.encode("utf-8", errors="ignore")
            return data, getattr(source, "name", "filelike.pdf")

        raise TypeError("Unsupported PDF source type.")

    def _basic_clean(self, text: str) -> str:
        text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
        text = text.replace("\r\n", "\n").replace("\r", "\n")
        text = re.sub(r"[ \t]+\n", "\n", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        text = re.sub(r"[ \t]{2,}", " ", text)
        return text.strip()

    def _remove_repetitive_headers_footers(self, pages: List[str]) -> List[str]:
        if not pages:
            return pages

        n_pages = len(pages)
        k = self.header_footer_lines
        if k <= 0:
            return pages

        header_counter = Counter()
        footer_counter = Counter()
        split_pages = []

        for t in pages:
            lines = [ln.strip() for ln in t.split("\n") if ln.strip()]
            split_pages.append(lines)
            for ln in lines[:k]:
                header_counter[ln] += 1
            for ln in lines[-k:]:
                footer_counter[ln] += 1

        min_count = max(2, int(self.header_footer_min_page_ratio * n_pages))
        header_to_remove = {ln for ln, c in header_counter.items() if c >= min_count}
        footer_to_remove = {ln for ln, c in footer_counter.items() if c >= min_count}

        cleaned_pages: List[str] = []
        for lines in split_pages:
            if not lines:
                cleaned_pages.append("")
                continue

            i = 0
            while i < len(lines) and lines[i] in header_to_remove:
                i += 1

            j = len(lines)
            while j > i and lines[j - 1] in footer_to_remove:
                j -= 1

            cleaned = "\n".join(lines[i:j]).strip()
            cleaned = self._basic_clean(cleaned)
            cleaned_pages.append(cleaned)

        return cleaned_pages


class Chunker:
    def __init__(
        self,
        *,
        max_chars: int = 1200,
        overlap_chars: int = 200,
        min_chunk_chars: int = 200,
    ) -> None:
        if overlap_chars >= max_chars:
            raise ValueError("overlap_chars must be < max_chars")
        self.max_chars = max_chars
        self.overlap_chars = overlap_chars
        self.min_chunk_chars = min_chunk_chars

    def chunk(self, doc: Document) -> List[Chunk]:
        text = (doc.text or "").strip()
        if not text:
            return []

        metadata_base: Dict[str, Any] = dict(doc.metadata or {})
        metadata_base["doc_id"] = doc.doc_id

        paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
        if not paragraphs:
            return []

        para_spans: List[Tuple[str, int, int]] = []
        cursor = 0
        for p in paragraphs:
            idx = text.find(p, cursor)
            if idx == -1:
                idx = text.find(p)
                if idx == -1:
                    continue
            start = idx
            end = idx + len(p)
            para_spans.append((p, start, end))
            cursor = end

        chunks: List[Chunk] = []
        i = 0
        chunk_idx = 0

        while i < len(para_spans):
            chunk_start = para_spans[i][1]
            chunk_text_parts = []
            chunk_end = chunk_start

            while i < len(para_spans):
                p, p_start, p_end = para_spans[i]
                tentative = ("\n\n".join(chunk_text_parts + [p])).strip()
                if chunk_text_parts and len(tentative) > self.max_chars:
                    break
                chunk_text_parts.append(p)
                chunk_end = p_end
                i += 1

            chunk_text = "\n\n".join(chunk_text_parts).strip()

            while len(chunk_text) < self.min_chunk_chars and i < len(para_spans):
                p, _, p_end = para_spans[i]
                extended = (chunk_text + "\n\n" + p).strip()
                if len(extended) > self.max_chars:
                    break
                chunk_text = extended
                chunk_end = p_end
                i += 1

            cmeta = dict(metadata_base)
            cmeta["chunk_index"] = chunk_idx
            cmeta["max_chars"] = self.max_chars
            cmeta["overlap_chars"] = self.overlap_chars

            chunk_id = f"{doc.doc_id}_c{chunk_idx}"
            chunks.append(
                Chunk(
                    chunk_id=chunk_id,
                    doc_id=doc.doc_id,
                    text=chunk_text,
                    metadata=cmeta,
                    start=chunk_start,
                    end=chunk_end,
                )
            )
            chunk_idx += 1

            if self.overlap_chars > 0 and i < len(para_spans):
                overlap_point = max(chunk_start, chunk_end - self.overlap_chars)
                rewind = i
                while rewind > 0 and para_spans[rewind - 1][1] >= overlap_point:
                    rewind -= 1
                i = rewind if rewind < i else i

        return chunks

In [4]:
file_path = "/Users/yanicrothlingshofer/Desktop/Telecom Paris/2A/RAG_WORKSHOP/sample2.pdf"
parser = PdfParser(per_page=True)
documents = parser.parse(file_path)
chunker = Chunker(max_chars=1200, overlap_chars=200, min_chunk_chars=200)
all_chunks = []
for doc in documents:
    chunks = chunker.chunk(doc)
    all_chunks.extend(chunks) 

print(f"Parsed {len(documents)} documents into {len(all_chunks)} chunks.")
print("Sample chunk metadata:", all_chunks[0].metadata if all_chunks else "No chunks created.")
print("Sample chunk text:", all_chunks[3].text[:200] + "..." if all_chunks else "No chunks created.")

Parsed 75 documents into 75 chunks.
Sample chunk metadata: {'source': 'sample2.pdf', 'file_hash': 'ec8f7074e1c1e3fcff3c325f4068e35c112688ae5fe968ba0cbbb0f92bb711ca', 'page': 1, 'num_pages': 75, 'is_scanned_like': False, 'doc_id': 'ec8f7074e1c1e3fcff3c325f4068e35c112688ae5fe968ba0cbbb0f92bb711ca_p1', 'chunk_index': 0, 'max_chars': 1200, 'overlap_chars': 200}
Sample chunk text: Généralités cliniques
Cancérologie générale
Chapitre 1
Généralités cliniques
1.1 Fréquence du cancer
1.1.1 Epidémiologie descriptive
1.1.1.1 Taux de mortalité
Le cancer est responsable de 26 % des déc...


In [5]:
# Embedding and vector storing

class Embedder:
    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def embed_texts(self, texts: List[str]) -> List[List[float]]:
        vectors = self.model.encode(texts, convert_to_numpy=True, show_progress_bar=False)
        return vectors.tolist()

    def embed_query(self, query: str) -> List[float]:
        vector = self.model.encode([query], convert_to_numpy=True, show_progress_bar=False)[0]
        return vector.tolist()


class VectorStore:
    def __init__(self, dim: int):
        self.index = faiss.IndexFlatIP(dim)
        self.chunks: List[Chunk] = []

    def upsert(self, chunks: List[Chunk], vectors: List[List[float]]) -> None:
        vecs = np.array(vectors).astype("float32")
        faiss.normalize_L2(vecs)
        self.index.add(vecs)
        self.chunks.extend(chunks)

    def search(
        self,
        query_vector: List[float],
        top_k: int,
        filters: Dict[str, Any],
    ) -> List[RetrievedChunk]:
        q = np.array([query_vector]).astype("float32")
        faiss.normalize_L2(q)
        scores, indices = self.index.search(q, top_k)

        results: List[RetrievedChunk] = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue
            chunk = self.chunks[idx]
            if filters:
                valid = all(chunk.metadata.get(k) == v for k, v in filters.items())
                if not valid:
                    continue
            results.append(
                RetrievedChunk(chunk=chunk, score=float(score), stage="dense")
            )
        return results


class SparseIndex:
    def __init__(self):
        self.bm25 = None
        self.chunks: List[Chunk] = []
        self.tokenized_corpus: List[List[str]] = []

    def upsert(self, chunks: List[Chunk]) -> None:
        self.chunks.extend(chunks)
        self.tokenized_corpus = [c.text.split() for c in self.chunks]
        self.bm25 = BM25Okapi(self.tokenized_corpus)

    def search(
        self,
        query: str,
        top_k: int,
        filters: Dict[str, Any],
    ) -> List[RetrievedChunk]:
        if not self.bm25:
            return []

        tokenized_query = query.split()
        scores = self.bm25.get_scores(tokenized_query)
        top_indices = np.argsort(scores)[::-1][:top_k]

        results: List[RetrievedChunk] = []
        for idx in top_indices:
            chunk = self.chunks[idx]
            if filters:
                valid = all(chunk.metadata.get(k) == v for k, v in filters.items())
                if not valid:
                    continue
            results.append(
                RetrievedChunk(chunk=chunk, score=float(scores[idx]), stage="sparse")
            )
        return results


class Reranker:
    def __init__(self, model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"):
        self.model = CrossEncoder(model_name)

    def rerank(
        self,
        query: str,
        candidates: List[RetrievedChunk],
        top_k: int,
    ) -> List[RetrievedChunk]:
        if not candidates:
            return []

        pairs = [(query, c.chunk.text) for c in candidates]
        scores = self.model.predict(pairs)

        ranked = sorted(
            zip(candidates, scores),
            key=lambda x: x[1],
            reverse=True,
        )

        return [
            RetrievedChunk(chunk=c.chunk, score=float(score), stage="rerank")
            for c, score in ranked[:top_k]
        ]
    
    

In [6]:
embedder = Embedder("sentence-transformers/all-MiniLM-L6-v2")

texts = [c.text for c in all_chunks]
vectors = embedder.embed_texts(texts)

dim = len(vectors[0])
vector_store = VectorStore(dim)
vector_store.upsert(all_chunks, vectors)

sparse_index = SparseIndex()
sparse_index.upsert(all_chunks)

reranker = Reranker("cross-encoder/ms-marco-MiniLM-L-6-v2")


query = "De quoi parle ce document ? Donne les points clés."
filters: Dict[str, Any] = {}  

qvec = embedder.embed_query(query)

dense_results = vector_store.search(qvec, top_k=10, filters=filters)
sparse_results = sparse_index.search(query, top_k=10, filters=filters)

by_id = {}
for r in dense_results + sparse_results:
    by_id[r.chunk.chunk_id] = r
candidates = list(by_id.values())

top = reranker.rerank(query, candidates, top_k=5)

for i, r in enumerate(top, 1):
    print(f"\n--- TOP {i} | {r.stage} | score={r.score:.4f} ---")
    print(r.chunk.metadata)
    print(r.chunk.text[:500])



context = "\n\n".join(
    [f"[{i}] (page {r.chunk.metadata.get('page')})\n{r.chunk.text}" for i, r in enumerate(top, 1)]
)

prompt = f"""Tu es un assistant.
Règles:
- Réponds uniquement à partir du CONTEXTE.
- Si tu ne trouves pas l'info dans le contexte, dis "Je ne sais pas d'après les sources fournies".
- Cite tes sources avec [1], [2], etc.

CONTEXTE:
{context}

QUESTION:
{query}

RÉPONSE:
"""


--- TOP 1 | rerank | score=-1.7493 ---
{'source': 'sample2.pdf', 'file_hash': 'ec8f7074e1c1e3fcff3c325f4068e35c112688ae5fe968ba0cbbb0f92bb711ca', 'page': 11, 'num_pages': 75, 'is_scanned_like': False, 'doc_id': 'ec8f7074e1c1e3fcff3c325f4068e35c112688ae5fe968ba0cbbb0f92bb711ca_p11', 'chunk_index': 0, 'max_chars': 1200, 'overlap_chars': 200}
Généralités cliniques
1.2.2 Le diagnostic d’extension
Le bilan d’extension par la clinique, l’imagerie, l’endoscopie et les constatations chirurgicales a
une importance capitale pour le pronostic et le traitement.
Cliniquement on précise le siège de la tumeur et les dimensions en cm en faisant des schémas et
des photographies si possible. Parfois la distinction entre infiltration inflammatoire et infiltration
tumorale est difficile à faire. Un réexamen après un court traitement antibiotique et a

--- TOP 2 | rerank | score=-2.0552 ---
{'source': 'sample2.pdf', 'file_hash': 'ec8f7074e1c1e3fcff3c325f4068e35c112688ae5fe968ba0cbbb0f92bb711ca', 'page': 1

In [ ]:
# LLM inference 

class LLM:
    def __init__(
        self,
        model_name: str = "Qwen/Qwen2.5-3B-Instruct",
        device: Optional[str] = None,
    ):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
        )
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        if self.device == "cpu":
            self.model.to("cpu")

    @torch.inference_mode()
    def generate(
        self,
        prompt: str,
        max_new_tokens: int = 120,
        temperature: float = 0,
        top_p: float = 0.9,
        do_sample: Optional[bool] = None,
        **kwargs: Any,
    ) -> str:
        if do_sample is None:
            do_sample = temperature > 0

        inputs = self.tokenizer(prompt, return_tensors="pt")
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}

        out = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=do_sample,
            eos_token_id=self.tokenizer.eos_token_id,
            **kwargs,
        )

        return self.tokenizer.decode(out[0], skip_special_tokens=True)

In [12]:
llm = LLM("Qwen/Qwen2.5-3B-Instruct")

prompt = """
Explique en 3 phrases ce qu'est le Retrieval Augmented Generation.
"""

response = llm.generate(prompt)

print(response)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

/opt/anaconda3/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


KeyboardInterrupt: 

In [46]:

class Guardrails:
    def __init__(
        self,
        *,
        min_contexts: int = 1,
        max_contexts: int = 8,
        min_score: float | None = None,
        require_citations: bool = True,
        deny_injection_patterns: bool = True,
    ) -> None:
        self.min_contexts = min_contexts
        self.max_contexts = max_contexts
        self.min_score = min_score
        self.require_citations = require_citations
        self.deny_injection_patterns = deny_injection_patterns

        self._inj_re = re.compile(
            r"(ignore (all|previous) instructions|system prompt|developer message|"
            r"reveal .*prompt|jailbreak|do anything now|DAN|"
            r"exfiltrate|password|secret|apikey|api key|token)",
            re.IGNORECASE,
        )

    def validate_context(self, query: str, contexts: List[RetrievedChunk]) -> List[RetrievedChunk]:
        filtered = contexts

        if self.min_score is not None:
            filtered = [c for c in filtered if c.score >= self.min_score]

        if self.deny_injection_patterns:
            filtered = [c for c in filtered if not self._inj_re.search(c.chunk.text or "")]

        filtered = filtered[: self.max_contexts]

        if len(filtered) < self.min_contexts:
            return []

        return filtered

    def validate_answer(self, query: str, answer: str, contexts: List[RetrievedChunk]) -> str:
        if not contexts:
            return "Je ne sais pas d'après les sources fournies."

        out = (answer or "").strip()
        if not out:
            return "Je ne sais pas d'après les sources fournies."

        if self.require_citations:
            if not re.search(r"\[\d+\]", out):
                return "Je ne sais pas d'après les sources fournies."

        if self.deny_injection_patterns and self._inj_re.search(out):
            return "Je ne sais pas d'après les sources fournies."

        return out


In [57]:
# Offline part for ingestion and indexing


@dataclass
class IndexingConfig:
    chunk_batch_size: int = 256
    embed_batch_size: int = 64


class Indexer:
    def __init__(
        self,
        parser: Parser,
        chunker: Chunker,
        embedder: Embedder,
        vector_store: VectorStore,
        sparse_index: Optional[SparseIndex] = None,
        cfg: IndexingConfig = IndexingConfig(),
    ) -> None:
        self.parser = parser
        self.chunker = chunker
        self.embedder = embedder
        self.vector_store = vector_store
        self.sparse_index = sparse_index
        self.cfg = cfg

    def ingest(self, sources: Iterable[Any]) -> int:
        all_chunks: List[Chunk] = []

        for src in sources:
            docs = self.parser.parse(src)
            for doc in docs:
                all_chunks.extend(self.chunker.chunk(doc))

        if not all_chunks:
            return 0

        total = 0

        for i in range(0, len(all_chunks), self.cfg.chunk_batch_size):
            batch = all_chunks[i : i + self.cfg.chunk_batch_size]
            texts = [c.text for c in batch]

            vectors: List[List[float]] = []
            for j in range(0, len(texts), self.cfg.embed_batch_size):
                sub = texts[j : j + self.cfg.embed_batch_size]
                vectors.extend(self.embedder.embed_texts(sub))

            self.vector_store.upsert(batch, vectors)

            if self.sparse_index:
                self.sparse_index.upsert(batch)

            total += len(batch)

        return total


In [58]:
class PromptBuilder:
    """
    Keep this isolated: prompt format evolves a lot (citations, JSON output, tools, etc.)
    """
    def build(self, query: str, contexts: List[RetrievedChunk]) -> str:
        context_lines = []
        for i, rc in enumerate(contexts, start=1):
            src = rc.chunk.metadata.get("source", "unknown")
            context_lines.append(f"[{i}] source={src} score={rc.score:.3f}\n{rc.chunk.text[:500]}\n")

        joined = "\n".join(context_lines)
        return f"""
            You are an assistant for industrial knowledge work.

            STRICT RULES:
            - Answer ONLY using the provided CONTEXT.
            - If the answer is not in the context, say: "I don't know from the provided sources."
            - Cite sources as [1], [2], etc.
            - Be concise and factual.

            CONTEXT:
            {joined}

            QUESTION:
            {query}

            ANSWER:
            """.strip()


In [59]:
@dataclass(frozen=True)
class RagAnswer:
    answer: str
    citations: List[Dict[str, Any]]
    debug: Dict[str, Any]


@dataclass
class QueryConfig:
    top_k_dense: int = 20
    top_k_sparse: int = 20
    top_k_rerank: int = 3
    use_hybrid: bool = True
    llm_max_tokens: int = 400
    llm_temperature: float = 0.2


class RagPipeline:
    def __init__(
        self,
        embedder: Embedder,
        vector_store: VectorStore,
        llm: LLM,
        prompt_builder: "PromptBuilder",
        reranker: Optional[Reranker] = None,
        sparse_index: Optional[SparseIndex] = None,
        guardrails: Optional[Guardrails] = None,
        cfg: QueryConfig = QueryConfig(),
    ) -> None:
        self.embedder = embedder
        self.vector_store = vector_store
        self.sparse_index = sparse_index
        self.reranker = reranker
        self.llm = llm
        self.prompt_builder = prompt_builder
        self.guardrails = guardrails or Guardrails()
        self.cfg = cfg

    def answer(self, query: str, filters: Optional[Dict[str, Any]] = None) -> RagAnswer:
        request_id = str(uuid.uuid4())
        filters = filters or {}
        t0 = time.time()

        qvec = self.embedder.embed_query(query)
        dense = self.vector_store.search(qvec, top_k=self.cfg.top_k_dense, filters=filters)

        sparse: List[RetrievedChunk] = []
        if self.cfg.use_hybrid and self.sparse_index:
            sparse = self.sparse_index.search(query, top_k=self.cfg.top_k_sparse, filters=filters)

        merged = self._merge_and_dedupe(dense, sparse)
        merged = self.guardrails.validate_context(query, merged)

        if self.reranker:
            ranked = self.reranker.rerank(query, merged, top_k=self.cfg.top_k_rerank)
        else:
            ranked = merged[: self.cfg.top_k_rerank]

        prompt = self.prompt_builder.build(query, ranked)

        raw = self.llm.generate(
            prompt,
            max_new_tokens=self.cfg.llm_max_tokens,
            temperature=self.cfg.llm_temperature,
        )

        final_answer = self.guardrails.validate_answer(query, raw, ranked)
        citations = self._build_citations(ranked)

        latency = time.time() - t0
        debug = {
            "request_id": request_id,
            "latency_s": latency,
            "dense_n": len(dense),
            "sparse_n": len(sparse),
            "merged_n": len(merged),
            "ranked_n": len(ranked),
        }

        return RagAnswer(answer=final_answer, citations=citations, debug=debug)

    def _merge_and_dedupe(
        self,
        dense: List[RetrievedChunk],
        sparse: List[RetrievedChunk],
    ) -> List[RetrievedChunk]:
        by_id: Dict[str, RetrievedChunk] = {}
        for r in dense + sparse:
            cid = r.chunk.chunk_id
            if cid not in by_id:
                by_id[cid] = r
        return list(by_id.values())

    def _build_citations(self, ranked: List[RetrievedChunk]) -> List[Dict[str, Any]]:
        cites = []
        for i, r in enumerate(ranked, start=1):
            m = r.chunk.metadata
            cites.append(
                {
                    "n": i,
                    "chunk_id": r.chunk.chunk_id,
                    "doc_id": r.chunk.doc_id,
                    "source": m.get("source"),
                    "score": r.score,
                    "start": r.chunk.start,
                    "end": r.chunk.end,
                }
            )
        return cites


In [50]:
@dataclass(frozen=True)
class EvalSample:
    query: str
    expected_answer: Optional[str] = None
    expected_sources: Optional[List[str]] = None


class Evaluator:
    def __init__(self, pipeline: RagPipeline) -> None:
        self.pipeline = pipeline

    def run(self, samples: List[EvalSample]) -> Dict[str, Any]:
        return {"n": len(samples)}

In [60]:
def build_app(
    *,
    embedder_model: str = "sentence-transformers/all-MiniLM-L6-v2",
    reranker_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2",
    llm_model: str = "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    per_page: bool = True,
    chunk_max_chars: int = 1200,
    chunk_overlap_chars: int = 200,
    chunk_min_chars: int = 200,
) -> Tuple[Indexer, RagPipeline]:
    parser = PdfParser(per_page=per_page)
    chunker = Chunker(
        max_chars=chunk_max_chars,
        overlap_chars=chunk_overlap_chars,
        min_chunk_chars=chunk_min_chars,
    )

    embedder = Embedder(embedder_model)
    dim = len(embedder.embed_query("dim"))

    vector_store = VectorStore(dim)
    sparse_index = SparseIndex()
    reranker = Reranker(reranker_model)
    llm = LLM(llm_model)

    prompt_builder = PromptBuilder()
    guardrails = Guardrails()

    indexer = Indexer(
        parser=parser,
        chunker=chunker,
        embedder=embedder,
        vector_store=vector_store,
        sparse_index=sparse_index,
    )

    rag = RagPipeline(
        embedder=embedder,
        vector_store=vector_store,
        llm=llm,
        prompt_builder=prompt_builder,
        reranker=reranker,
        sparse_index=sparse_index,
        guardrails=guardrails,
    )

    return indexer, rag



In [61]:
# Test with a sample PDF and queries

indexer, rag = build_app()

indexer.ingest(["sample2.pdf"])

print(rag.answer("De quoi parle ce document ?").answer)

You are an assistant for industrial knowledge work.

            STRICT RULES:
            - Answer ONLY using the provided CONTEXT.
            - If the answer is not in the context, say: "I don't know from the provided sources."
            - Cite sources as [1], [2], etc.
            - Be concise and factual.

            CONTEXT:
            [1] source=sample2.pdf score=-5.722
Prévention, dépistage, cancers professionnels
[Tapez ici]
inscrits sur les tableaux (par exemples les brouillards ou vapeurs d’acide sulfurique pur ou en
mélange), ou tout autre situation où l’on suspecte médicalement une relation entre un éventuel
cancérogène, un cancer et une activité professionnelle particulière.
Toutes ces déclarations permettent non seulement d’aider les victimes mais aussi d’entreprendre
des actions de prévention particulièrement utiles. De plus ces déclarations permettent

[2] source=sample2.pdf score=-6.458
Généralités cliniques
1.9 Les essais randomisés
Lorsqu’il n’y a pas de supério

In [ ]:

@dataclass(frozen=True)
class RetrievedChunk:
    chunk: Chunk
    score: float
    stage: str = 'dense'


@dataclass(frozen=True)
class RagAnswer:
    answer: str
    citations: List[Dict[str, Any]]  
    debug: Dict[str, Any]    



# =========================
# 1) Interfaces (replaceable)
# =========================

class Parser(Protocol):
    def parse(self, source: Any) -> List[Document]:
        ...


class Chunker(Protocol):
    def chunk(self, doc: Document) -> List[Chunk]:
        ...


class Embedder(Protocol):
    def embed_texts(self, texts: List[str]) -> List[List[float]]:
        ...

    def embed_query(self, query: str) -> List[float]:
        ...


class VectorStore(Protocol):
    def upsert(self, chunks: List[Chunk], vectors: List[List[float]]) -> None:
        ...

    def search(self, query_vector: List[float], top_k: int, filters: Dict[str, Any]) -> List[RetrievedChunk]:
        ...


class SparseIndex(Protocol):
    """BM25 or similar sparse retriever (optional)."""
    def upsert(self, chunks: List[Chunk]) -> None:
        ...

    def search(self, query: str, top_k: int, filters: Dict[str, Any]) -> List[RetrievedChunk]:
        ...


class Reranker(Protocol):
    def rerank(self, query: str, candidates: List[RetrievedChunk], top_k: int) -> List[RetrievedChunk]:
        ...


class LLM(Protocol):
    def generate(self, prompt: str, **kwargs: Any) -> str:
        ...


class Guardrails(Protocol):
    """Optional: safety checks, injection detection, policy enforcement."""
    def validate_context(self, query: str, contexts: List[RetrievedChunk]) -> List[RetrievedChunk]:
        ...

    def validate_answer(self, query: str, answer: str, contexts: List[RetrievedChunk]) -> str:
        ...


class Logger(Protocol):
    def log(self, event_name: str, payload: Dict[str, Any]) -> None:
        ...


# =========================
# 2) Concrete “empty” implementations (TODO)
# =========================

class NoopLogger:
    def log(self, event_name: str, payload: Dict[str, Any]) -> None:
        pass


class SimpleGuardrails:
    def validate_context(self, query: str, contexts: List[RetrievedChunk]) -> List[RetrievedChunk]:
        # TODO: filter untrusted sources, detect prompt injection patterns, enforce ACL
        return contexts

    def validate_answer(self, query: str, answer: str, contexts: List[RetrievedChunk]) -> str:
        # TODO: enforce "answer only from sources", check for missing citations, etc.
        return answer


# =========================
# 3) Indexing pipeline
# =========================

@dataclass
class IndexingConfig:
    chunk_batch_size: int = 256
    embed_batch_size: int = 64


class Indexer:
    def __init__(
        self,
        parser: Parser,
        chunker: Chunker,
        embedder: Embedder,
        vector_store: VectorStore,
        sparse_index: Optional[SparseIndex] = None,
        logger: Logger = NoopLogger(),
        cfg: IndexingConfig = IndexingConfig(),
    ) -> None:
        self.parser = parser
        self.chunker = chunker
        self.embedder = embedder
        self.vector_store = vector_store
        self.sparse_index = sparse_index
        self.logger = logger
        self.cfg = cfg

    def ingest(self, sources: Iterable[Any]) -> int:
        """
        End-to-end ingestion.
        Returns number of chunks indexed.
        """
        t0 = time.time()
        all_chunks: List[Chunk] = []

        # 1) Parse sources -> documents
        for src in sources:
            docs = self.parser.parse(src)
            for doc in docs:
                # 2) Chunk per doc
                chunks = self.chunker.chunk(doc)
                all_chunks.extend(chunks)

        if not all_chunks:
            self.logger.log("index.ingest.empty", {"sources": str(sources)})
            return 0

        # 3) Embed + upsert in batches
        total = 0
        for i in range(0, len(all_chunks), self.cfg.chunk_batch_size):
            batch = all_chunks[i : i + self.cfg.chunk_batch_size]
            texts = [c.text for c in batch]

            # embed in smaller sub-batches if needed
            vectors: List[List[float]] = []
            for j in range(0, len(texts), self.cfg.embed_batch_size):
                sub = texts[j : j + self.cfg.embed_batch_size]
                vectors.extend(self.embedder.embed_texts(sub))

            # 4) upsert dense
            self.vector_store.upsert(batch, vectors)

            # 5) upsert sparse (optional)
            if self.sparse_index:
                self.sparse_index.upsert(batch)

            total += len(batch)

        self.logger.log("index.ingest.done", {
            "chunks_indexed": total,
            "latency_s": time.time() - t0
        })
        return total


# =========================
# 4) Retrieval + Generation pipeline
# =========================

@dataclass
class QueryConfig:
    top_k_dense: int = 20
    top_k_sparse: int = 20
    top_k_rerank: int = 6
    use_hybrid: bool = True

    llm_max_tokens: int = 400
    llm_temperature: float = 0.2


class RagPipeline:
    def __init__(
        self,
        embedder: Embedder,
        vector_store: VectorStore,
        llm: LLM,
        prompt_builder: "PromptBuilder",
        reranker: Optional[Reranker] = None,
        sparse_index: Optional[SparseIndex] = None,
        guardrails: Optional[Guardrails] = None,
        logger: Logger = NoopLogger(),
        cfg: QueryConfig = QueryConfig(),
    ) -> None:
        self.embedder = embedder
        self.vector_store = vector_store
        self.sparse_index = sparse_index
        self.reranker = reranker
        self.llm = llm
        self.prompt_builder = prompt_builder
        self.guardrails = guardrails or SimpleGuardrails()
        self.logger = logger
        self.cfg = cfg

    def answer(self, query: str, filters: Optional[Dict[str, Any]] = None) -> RagAnswer:
        """
        Query-time pipeline:
        - embed query
        - retrieve dense (+ sparse optional)
        - merge + dedupe
        - guardrails (context)
        - rerank
        - build prompt
        - generate
        - postprocess (citations, checks)
        """
        request_id = str(uuid.uuid4())
        filters = filters or {}
        t0 = time.time()

        # 1) Dense retrieval
        qvec = self.embedder.embed_query(query)
        dense = self.vector_store.search(qvec, top_k=self.cfg.top_k_dense, filters=filters)

        # 2) Sparse retrieval (optional)
        sparse: List[RetrievedChunk] = []
        if self.cfg.use_hybrid and self.sparse_index:
            sparse = self.sparse_index.search(query, top_k=self.cfg.top_k_sparse, filters=filters)

        # 3) Merge + dedupe (by chunk_id)
        merged = self._merge_and_dedupe(dense, sparse)

        # 4) Context guardrails (ACL, injection detection, etc.)
        merged = self.guardrails.validate_context(query, merged)

        # 5) Rerank (optional but recommended)
        if self.reranker:
            ranked = self.reranker.rerank(query, merged, top_k=self.cfg.top_k_rerank)
        else:
            ranked = merged[: self.cfg.top_k_rerank]

        # 6) Build prompt
        prompt = self.prompt_builder.build(query, ranked)

        # 7) Generate
        raw = self.llm.generate(
            prompt,
            max_new_tokens=self.cfg.llm_max_tokens,
            temperature=self.cfg.llm_temperature,
        )

        # 8) Postprocess + enforce answer policies
        final_answer = self.guardrails.validate_answer(query, raw, ranked)
        citations = self._build_citations(ranked)

        latency = time.time() - t0
        debug = {
            "request_id": request_id,
            "latency_s": latency,
            "dense_n": len(dense),
            "sparse_n": len(sparse),
            "merged_n": len(merged),
            "ranked_n": len(ranked),
        }

        self.logger.log("rag.answer.done", debug)
        return RagAnswer(answer=final_answer, citations=citations, debug=debug)

    def _merge_and_dedupe(self, dense: List[RetrievedChunk], sparse: List[RetrievedChunk]) -> List[RetrievedChunk]:
        # TODO: smarter fusion (Reciprocal Rank Fusion, score normalization, etc.)
        by_id: Dict[str, RetrievedChunk] = {}
        for r in dense + sparse:
            cid = r.chunk.chunk_id
            if cid not in by_id:
                by_id[cid] = r
        return list(by_id.values())

    def _build_citations(self, ranked: List[RetrievedChunk]) -> List[Dict[str, Any]]:
        cites = []
        for i, r in enumerate(ranked, start=1):
            m = r.chunk.metadata
            cites.append({
                "n": i,
                "chunk_id": r.chunk.chunk_id,
                "doc_id": r.chunk.doc_id,
                "source": m.get("source"),
                "score": r.score,
                "start": r.chunk.start,
                "end": r.chunk.end,
            })
        return cites


# =========================
# 5) Prompt building
# =========================

class PromptBuilder:
    """
    Keep this isolated: prompt format evolves a lot (citations, JSON output, tools, etc.)
    """
    def build(self, query: str, contexts: List[RetrievedChunk]) -> str:
        context_lines = []
        for i, rc in enumerate(contexts, start=1):
            src = rc.chunk.metadata.get("source", "unknown")
            context_lines.append(f"[{i}] source={src} score={rc.score:.3f}\n{rc.chunk.text}\n")

        joined = "\n".join(context_lines)
        return f"""
            You are an assistant for industrial knowledge work.

            STRICT RULES:
            - Answer ONLY using the provided CONTEXT.
            - If the answer is not in the context, say: "I don't know from the provided sources."
            - Cite sources as [1], [2], etc.
            - Be concise and factual.

            CONTEXT:
            {joined}

            QUESTION:
            {query}

            ANSWER:
            """.strip()


# =========================
# 6) Evaluation hooks (offline)
# =========================

@dataclass(frozen=True)
class EvalSample:
    query: str
    expected_answer: Optional[str] = None
    expected_sources: Optional[List[str]] = None  # doc_ids or sources


class Evaluator:
    """
    Offline evaluation skeleton:
    - retrieval metrics: precision@k, recall@k, MRR
    - answer metrics: groundedness, faithfulness, exact match (if possible)
    """
    def __init__(self, pipeline: RagPipeline, logger: Logger = NoopLogger()) -> None:
        self.pipeline = pipeline
        self.logger = logger

    def run(self, samples: List[EvalSample]) -> Dict[str, Any]:
        # TODO: implement and store results (CSV/JSON)
        results = {"n": len(samples)}
        return results


# =========================
# 7) Wiring (composition root)
# =========================

def build_app() -> Tuple[Indexer, RagPipeline]:
    """
    Assemble real implementations here:
    - Parser: PDF/HTML/Confluence/Jira...
    - Chunker: token-based with structure awareness
    - Embedder: sentence-transformers / HF embeddings
    - VectorStore: FAISS/Qdrant/Weaviate/OpenSearch
    - SparseIndex: BM25 (optional)
    - Reranker: cross-encoder (optional)
    - LLM: HF transformers / vLLM / TGI
    """

    # TODO: replace all stubs with real implementations
    parser = ...        # implements Parser
    chunker = ...       # implements Chunker
    embedder = ...      # implements Embedder
    vector_store = ...  # implements VectorStore
    sparse_index = None # or implements SparseIndex
    reranker = None     # or implements Reranker
    llm = ...           # implements LLM
    prompt_builder = PromptBuilder()
    logger = NoopLogger()

    indexer = Indexer(
        parser=parser,
        chunker=chunker,
        embedder=embedder,
        vector_store=vector_store,
        sparse_index=sparse_index,
        logger=logger,
    )

    rag = RagPipeline(
        embedder=embedder,
        vector_store=vector_store,
        sparse_index=sparse_index,
        reranker=reranker,
        llm=llm,
        prompt_builder=prompt_builder,
        guardrails=SimpleGuardrails(),
        logger=logger,
    )

    return indexer, rag


# =========================
# 8) Example usage
# =========================

def main() -> None:
    indexer, rag = build_app()

    # Indexing
    sources = [
        # TODO: PDF paths, URLs, DB queries, S3 keys, Confluence pages...
    ]
    # indexer.ingest(sources)

    # Query
    # ans = rag.answer("What is our password rotation policy?", filters={"acl_group": "IT"})
    # print(ans.answer)
    # print(ans.citations)
    # print(ans.debug)


if __name__ == "__main__":
    main()